# EDA Fourcasters

Ce notebook sert à faire un premier état des lieux des tables produites par dbt.
On regarde les volumes, les dates, les valeurs manquantes et quelques
répartitions utiles pour préparer Power BI et un futur modèle de ML.

> Attention : le niveau Météo-France est un **danger prévu**, pas un relevé
> du nombre de feux réellement observés. La table ML contient donc des
> variables candidates, mais pas encore de cible d'apprentissage.

In [ ]:
import pandas as pd
from google.cloud import bigquery

PROJET_GCP = "fourcasters-openmeteo-loick"
DATASET = "openmeteo_analyse"
client = bigquery.Client(project=PROJET_GCP)

def lire_requete(sql):
    """Exécute une requête et renvoie un DataFrame."""
    return client.query(sql).to_dataframe()

## 1. Volumes et période couverte

In [ ]:
volumes = lire_requete(f"""
SELECT 'fact_meteo' AS table_nom, COUNT(*) AS lignes,
       MIN(date) AS date_min, MAX(date) AS date_max
FROM `{PROJET_GCP}.{DATASET}.fact_meteo`
UNION ALL
SELECT 'fact_danger_incendie', COUNT(*), MIN(date_publication), MAX(date_publication)
FROM `{PROJET_GCP}.{DATASET}.fact_danger_incendie`
UNION ALL
SELECT 'pbi_risque_incendie', COUNT(*), MIN(date_publication), MAX(date_publication)
FROM `{PROJET_GCP}.{DATASET}.pbi_risque_incendie`
UNION ALL
SELECT 'ml_features_incendie', COUNT(*), MIN(date_publication), MAX(date_publication)
FROM `{PROJET_GCP}.{DATASET}.ml_features_incendie`
ORDER BY table_nom
""")
volumes

## 2. Valeurs manquantes dans la table Power BI

In [ ]:
manquants = lire_requete(f"""
SELECT
  COUNT(*) AS lignes,
  COUNTIF(NOT meteo_disponible) AS lignes_sans_meteo,
  ROUND(100 * COUNTIF(NOT meteo_disponible) / COUNT(*), 2) AS pct_sans_meteo,
  COUNTIF(temperature_moyenne IS NULL) AS temperature_manquante,
  COUNTIF(precipitations_totales IS NULL) AS precipitation_manquante
FROM `{PROJET_GCP}.{DATASET}.pbi_risque_incendie`
""")
manquants

## 3. Quelques indicateurs météo

In [ ]:
resume_meteo = lire_requete(f"""
SELECT
  EXTRACT(YEAR FROM date) AS annee,
  ROUND(AVG(temperature_moyenne), 1) AS temperature_moyenne,
  ROUND(MAX(temperature_maximale), 1) AS temperature_maximale,
  ROUND(SUM(precipitations_totales), 1) AS precipitations_totales
FROM `{PROJET_GCP}.{DATASET}.fact_meteo`
GROUP BY annee
ORDER BY annee
""")
resume_meteo

## 4. Répartition du danger prévu

In [ ]:
repartition_danger = lire_requete(f"""
SELECT
  echeance,
  niveau_danger,
  COUNT(*) AS lignes,
  ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY echeance), 2) AS pct
FROM `{PROJET_GCP}.{DATASET}.fact_danger_incendie`
GROUP BY echeance, niveau_danger
ORDER BY echeance, niveau_danger
""")
repartition_danger

## 5. Première lecture

- `pbi_risque_incendie` peut être reliée à `dim_date` et `dim_departement`
  dans Power BI.
- `ml_features_incendie` est au grain publication-département. Les variables
  sur 7 jours s'arrêtent à la veille de la publication pour limiter la fuite
  d'information.
- Pour entraîner un modèle supervisé, il faudra ajouter une source de feux
  observés (date, département et éventuellement surface ou gravité), puis
  construire une cible sans utiliser d'information future.

Les résultats chiffrés doivent être relus après chaque import d'archives :
les données Météo-France ne couvrent pas nécessairement la même période que
les observations Open-Meteo.

In [ ]:
# Graphiques facultatifs : matplotlib n'est pas nécessaire pour le pipeline quotidien.
try:
    import matplotlib.pyplot as plt
except ImportError:
    print("Installe matplotlib pour afficher les graphiques : uv add matplotlib")
else:
    resume_meteo.plot(x="annee", y="temperature_moyenne", marker="o")
    plt.title("Température moyenne par année")
    plt.ylabel("Température (°C)")
    plt.show()

    repartition_danger.pivot(index="niveau_danger", columns="echeance", values="pct").plot(kind="bar")
    plt.title("Répartition du danger prévu")
    plt.ylabel("Part des lignes (%)")
    plt.show()